<a href="https://colab.research.google.com/github/edgarcia2022-a11y/Hystrix/blob/main/Ed_Garcia_ML_Basics_Customer_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Kaggle Data
# Source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# --- Start Fix: Adjusting features for the Telco Customer Churn dataset ---

# Define appropriate numerical and categorical features from the dataset
numerical_features = ['tenure', 'MonthlyCharges']
categorical_features = ['gender', 'Partner', 'Dependents', 'SeniorCitizen', 'InternetService', 'Contract']

# Features and target
X = df[numerical_features + categorical_features]
y = df['Churn'] # Corrected: Assuming 'Churn' (uppercase C) is the correct target column name

# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
    ])

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear')) # Added solver for robustness
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
# Update new_customer to match the selected features
new_customer_data = {
    'tenure': [24],
    'MonthlyCharges': [55.0],
    'gender': ['Male'],
    'Partner': ['Yes'],
    'Dependents': ['No'],
    'SeniorCitizen': [0],
    'InternetService': ['DSL'],
    'Contract': ['One year']
}
new_customer = pd.DataFrame(new_customer_data)

churn_probability = model.predict_proba(new_customer)[0][1] # Probability of churn

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0
print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients
# Ensure get_feature_names_out uses categorical_features and feature_names includes numerical_features
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

# Model predicts that the new customer has a 14% probability of churning, not likely to churn

Churn Probability for new customer: 0.14
Churn Prediction (1 = churn, 0 = no churn): 0

Model Coefficients:
num__tenure: -0.78
num__MonthlyCharges: 0.14
cat__gender_Female: -0.25
cat__gender_Male: -0.29
cat__Partner_No: -0.30
cat__Partner_Yes: -0.24
cat__Dependents_No: -0.15
cat__Dependents_Yes: -0.39
cat__SeniorCitizen_0: -0.42
cat__SeniorCitizen_1: -0.12
cat__InternetService_DSL: -0.21
cat__InternetService_Fiber optic: 0.72
cat__InternetService_No: -1.05
cat__Contract_Month-to-month: 0.64
cat__Contract_One year: -0.16
cat__Contract_Two year: -1.02
